# Sentiment Analysis con Transformer (mejorado: `<UNK>` y dataset ampliado)

Este notebook introduce un pipeline sencillo para **clasificación de sentimientos** usando un Transformer en PyTorch.

Mejoras incluidas:
- Token especial `<PAD>` = 0 y `<UNK>` = 1 (que maneja palabras desconocidas).
- Dataset simulado más grande y diverso para mejorar generalización.
- Pipeline completo: generación de datos, vocab, dataset, modelo, entrenamiento y predicción.

Es un ejemplo educativo pensado para ejecutarse en CPU en pocos minutos (dependiendo de tu máquina).

In [1]:
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import numpy as np
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


## 1) Generar dataset simulado ampliado

In [2]:
positive = [
    "i love this movie",
    "this was fantastic",
    "great experience",
    "absolutely wonderful",
    "what a masterpiece",
    "brilliant work",
    "i enjoyed it a lot",
    "highly recommend this film",
    "truly amazing",
    "loved every minute",
    "wonderful acting",
    "captivating story",
    "beautiful cinematography",
    "incredible soundtrack",
    "a delightful experience",
]

negative = [
    "i hate this movie",
    "this was terrible",
    "awful experience",
    "absolutely horrible",
    "what a disaster",
    "very boring",
    "waste of time",
    "worst film ever",
    "hated every minute",
    "poor acting",
    "predictable plot",
    "forgettable soundtrack",
    "nothing interesting",
    "a total disappointment",
]

# Create variations by adding small adjectives or short phrases
extras = ['really', 'very', 'so', 'absolutely', 'truly', 'extremely', 'quite']
data = []
for _ in range(1000):
    if random.random() < 0.5:
        base = random.choice(positive)
        if random.random() < 0.5:
            base = random.choice(extras) + ' ' + base
        data.append((base, 1))
    else:
        base = random.choice(negative)
        if random.random() < 0.5:
            base = random.choice(extras) + ' ' + base
        data.append((base, 0))

random.shuffle(data)
print('Primeros 8 ejemplos:')
for i in range(8):
    print(data[i])


Primeros 8 ejemplos:
('quite i love this movie', 1)
('very highly recommend this film', 1)
('truly predictable plot', 0)
('really truly amazing', 1)
('absolutely horrible', 0)
('extremely waste of time', 0)
('beautiful cinematography', 1)
('quite i enjoyed it a lot', 1)


## 2) Construir vocabulario con `<PAD>` y `<UNK>`

In [3]:
PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'

vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for text, _ in data:
    for w in text.lower().split():
        if w not in vocab:
            vocab[w] = len(vocab)
vocab_size = len(vocab)
print('Tamaño vocab (incluye PAD y UNK):', vocab_size)
list(vocab.items())[:12]


Tamaño vocab (incluye PAD y UNK): 61


[('<PAD>', 0),
 ('<UNK>', 1),
 ('quite', 2),
 ('i', 3),
 ('love', 4),
 ('this', 5),
 ('movie', 6),
 ('very', 7),
 ('highly', 8),
 ('recommend', 9),
 ('film', 10),
 ('truly', 11)]

## 3) Dataset y DataLoader (mapeo de `<UNK>`)

In [4]:
class TextDataset(Dataset):
    def __init__(self, data, vocab, max_len=8):
        self.data = data
        self.vocab = vocab
        self.max_len = max_len
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        text, label = self.data[idx]
        tokens = [self.vocab.get(w, self.vocab[UNK_TOKEN]) for w in text.lower().split()]
        if len(tokens) < self.max_len:
            tokens = tokens + [self.vocab[PAD_TOKEN]] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
        return torch.tensor(tokens, dtype=torch.long), torch.tensor(label, dtype=torch.long)

dataset = TextDataset(data, vocab, max_len=8)
train_size = int(0.9 * len(dataset))
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, len(dataset) - train_size])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
print('Tamaño train:', len(train_dataset), 'val:', len(val_dataset))


Tamaño train: 900 val: 100


## 4) Modelo Transformer (clasificador)

In [5]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, num_heads=4, hidden_dim=128, num_layers=2, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=hidden_dim, dropout=0.1)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(embed_dim, num_classes)
    def forward(self, x):
        # x: (batch, seq_len)
        emb = self.embedding(x)           # (batch, seq, embed)
        emb = emb.permute(1, 0, 2)        # (seq, batch, embed) for transformer
        enc = self.transformer(emb)      # (seq, batch, embed)
        pooled = enc.mean(dim=0)         # (batch, embed)
        out = self.fc(pooled)            # (batch, num_classes)
        return out

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TransformerClassifier(vocab_size).to(device)
print(model)


TransformerClassifier(
  (embedding): Embedding(61, 64, padding_idx=0)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (fc): Linear(in_features=64, out_features=2, bias=True)
)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


## 5) Entrenamiento (rápido) y validación

In [6]:
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
epochs = 8

for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for Xv, yv in val_loader:
            Xv = Xv.to(device)
            yv = yv.to(device)
            preds = model(Xv)
            predicted = preds.argmax(dim=1)
            correct += (predicted == yv).sum().item()
            total += yv.size(0)
    val_acc = correct / total
    print(f'Epoch {epoch+1}/{epochs} - train_loss: {avg_loss:.4f} - val_acc: {val_acc:.4f}')


Epoch 1/8 - train_loss: 0.3382 - val_acc: 1.0000
Epoch 2/8 - train_loss: 0.0087 - val_acc: 1.0000
Epoch 3/8 - train_loss: 0.0016 - val_acc: 1.0000
Epoch 4/8 - train_loss: 0.0011 - val_acc: 1.0000
Epoch 5/8 - train_loss: 0.0009 - val_acc: 1.0000
Epoch 6/8 - train_loss: 0.0008 - val_acc: 1.0000
Epoch 7/8 - train_loss: 0.0006 - val_acc: 1.0000
Epoch 8/8 - train_loss: 0.0005 - val_acc: 1.0000


## 6) Probar con la frase que fallaba antes

In [7]:
def predict_text(text, vocab, model, device, max_len=8):
    tokens = [vocab.get(w, vocab[UNK_TOKEN]) for w in text.lower().split()]
    if len(tokens) < max_len:
        tokens = tokens + [vocab[PAD_TOKEN]] * (max_len - len(tokens))
    else:
        tokens = tokens[:max_len]
    xt = torch.tensor([tokens], dtype=torch.long).to(device)
    model.eval()
    with torch.no_grad():
        out = model(xt)
        pred = out.argmax(dim=1).item()
    return pred

test_sentence = 'I really loved this amazing film'
pred = predict_text(test_sentence, vocab, model, device)
print(f"Frase: '{test_sentence}' -> Sentimiento: {'Positivo' if pred==1 else 'Negativo'}")


Frase: 'I really loved this amazing film' -> Sentimiento: Positivo
